In [1]:
print("Applying Python 3.12 ctypes array-handler patch...")
import ctypes
try:
    from OpenGL.arrays import arraydatatype
    from OpenGL.arrays import ctypesarrays
    
    CArgObject = type(ctypes.byref(ctypes.c_int(0)))
    CStructType = type(ctypes.c_int)
    
    orig_get_type_handler = arraydatatype.ArrayDatatype.get_type_handler
    
    def patched_get_type_handler(typ, *args, **kwargs):
        if typ is CArgObject:
            return orig_get_type_handler(ctypes.c_void_p)
        if isinstance(typ, CStructType) or typ is CStructType:
            return orig_get_type_handler(ctypes.c_void_p)
        return orig_get_type_handler(typ, *args, **kwargs)
        
    arraydatatype.ArrayDatatype.get_type_handler = patched_get_type_handler
    print("OpenGL patch applied successfully.")
except Exception as e:
    print(f"OpenGL patch warning: {e}")

Applying Python 3.12 ctypes array-handler patch...
OpenGL patch warning: type object 'ArrayDatatype' has no attribute 'get_type_handler'


In [2]:
import os
import json
import numpy as np

In [3]:
os.environ['PYRENDER_BACKEND'] = 'egl'

In [4]:
if 'DISPLAY' not in os.environ:
    print("Initializing background display buffer...")
    !apt-get update -y > /dev/null
    !apt-get install -y xvfb > /dev/null
    !pip install pyvirtualdisplay > /dev/null
    from pyvirtualdisplay import Display
    display = Display(visible=0, size=(1400, 900))
    display.start()
    print("Virtual display active.")

In [5]:
DATASET_DIR = "data/synthetic_dataset"
IMAGES_DIR = os.path.join(DATASET_DIR, "images")
MASKS_DIR = os.path.join(DATASET_DIR, "masks")
PROGRESS_PATH = os.path.join(DATASET_DIR, "progress.json")

os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(MASKS_DIR, exist_ok=True)

In [6]:
TOTAL_SAMPLES = 500

In [7]:
def load_progress():
    """Loads state tracking dictionary to avoid redundant generation passes."""
    if os.path.exists(PROGRESS_PATH):
        try:
            with open(PROGRESS_PATH, "r") as f:
                return json.load(f)
        except Exception as e:
            print(f"Warning reading progress tracker, starting fresh: {e}")
    return {"completed_ids": []}

In [8]:
def save_progress(completed_ids):
    """Saves the current generation state to disk."""
    with open(PROGRESS_PATH, "w") as f:
        json.dump({"completed_ids": sorted(list(set(completed_ids)))}, f, indent=2)

In [9]:
progress = load_progress()
completed_ids = progress["completed_ids"]
print(f"Progress loaded. Completed samples: {len(completed_ids)} / {TOTAL_SAMPLES}")

Progress loaded. Completed samples: 0 / 500


In [10]:
import rasterio
from pyproj import Transformer
from PIL import Image
import pyrender
import trimesh

In [11]:
print("Building terrain mesh...")

# 1. Load active DEM and detect CRS / Bounds dynamically
with rasterio.open("data/dem.tif") as src:
    dem_data = src.read(1).astype(np.float32)
    [pixel_width, row_rotation, start_x, col_rotation, pixel_height, start_y] = src.transform[:6]

    raw_xs = start_x + np.arange(src.width) * pixel_width
    raw_ys = start_y + np.arange(src.height) * pixel_height
    
    if pixel_height < 0:
        raw_ys = raw_ys[::-1]
        dem_data = np.flipud(dem_data)
        
    dem_crs = src.crs.to_string()

# 2. Local Viewport Cropping
center_x = raw_xs[len(raw_xs) // 2]
center_y = raw_ys[len(raw_ys) // 2]

crop_size_x = 110000.0  
crop_size_y = 110000.0

crop_min_x = center_x - crop_size_x / 2.0
crop_max_x = center_x + crop_size_x / 2.0
crop_min_y = center_y - crop_size_y / 2.0
crop_max_y = center_y + crop_size_y / 2.0

crop_mask_x = (raw_xs >= crop_min_x) & (raw_xs <= crop_max_x)
crop_mask_y = (raw_ys >= crop_min_y) & (raw_ys <= crop_max_y)

xs_cropped = raw_xs[crop_mask_x]
ys_cropped = raw_ys[crop_mask_y]
dem_data_cropped = dem_data[crop_mask_y][:, crop_mask_x]

# Apply Stride 2
stride = 2  
dem_data_final = dem_data_cropped[::stride, ::stride]
xs_final = xs_cropped[::stride]
ys_final = ys_cropped[::stride]
height, width = dem_data_final.shape

# Generate 3D Vertices
vertices = np.column_stack((np.meshgrid(xs_final.astype(np.float32), ys_final.astype(np.float32))[0].ravel(),
                            np.meshgrid(xs_final.astype(np.float32), ys_final.astype(np.float32))[1].ravel(),
                            dem_data_final.ravel()))

# Generate Mesh Topology
r, c = np.meshgrid(np.arange(height - 1, dtype=np.int32), np.arange(width - 1, dtype=np.int32), indexing='ij')
v0 = r * width + c
v1 = v0 + 1
v2 = (r + 1) * width + c
v3 = v2 + 1
faces = np.vstack((np.column_stack((v0.ravel(), v1.ravel(), v2.ravel())),
                    np.column_stack((v1.ravel(), v3.ravel(), v2.ravel()))))

# Compute Dynamic GPS Coordinates for Texturing
dem_to_gps = Transformer.from_crs(dem_crs, "EPSG:4326", always_xy=True)
gps_min_lon, gps_min_lat = dem_to_gps.transform(xs_final[0], ys_final[0])
gps_max_lon, gps_max_lat = dem_to_gps.transform(xs_final[-1], ys_final[-1])

# Load Satellite Texture from disk cache
texture_img_path = "data/satellite_texture_cropped.png"
bounds_path = "data/satellite_texture_bounds_cropped.npz"
sat_image = Image.open(texture_img_path)
bounds_data = np.load(bounds_path)
img_min_lat = float(bounds_data['min_lat'])
img_max_lat = float(bounds_data['max_lat'])
img_min_lon = float(bounds_data['min_lon'])
img_max_lon = float(bounds_data['max_lon'])

sat_array = np.array(sat_image)
stitched_h, stitched_w, _ = sat_array.shape

# Perform Planar UV Mapping
lon_v, lat_v = dem_to_gps.transform(vertices[:, 0], vertices[:, 1])
u = np.clip((lon_v - img_min_lon) / (img_max_lon - img_min_lon), 0.0, 1.0)
v = np.clip((lat_v - img_min_lat) / (img_max_lat - img_min_lat), 0.0, 1.0)
uv = np.column_stack((u, v))

# Bind Base Textures
base_color_tex = pyrender.Texture(source=sat_image, source_channels='RGB')
mr_array = np.zeros((stitched_h, stitched_w, 3), dtype=np.uint8)
snow_mask = (sat_array[:, :, 0] > 180) & (sat_array[:, :, 1] > 180) & (sat_array[:, :, 2] > 180)
mr_array[:, :, 1] = np.where(snow_mask, int(0.70 * 255), int(0.95 * 255)).astype(np.uint8)
mr_img = Image.fromarray(mr_array)
mr_mr_tex = pyrender.Texture(source=mr_img, source_channels='RGB')

# PBR Material with normalTexture=None to prevent Moire ground artifacts
pbr_material = pyrender.MetallicRoughnessMaterial(
    baseColorTexture=base_color_tex,
    normalTexture=None,  # Disabled to fix ground artifacts
    metallicRoughnessTexture=mr_mr_tex,
    roughnessFactor=1.0,
    metallicFactor=0.0
)

# Smooth 5x5 Analytical Vertex Normals
padded_dem = np.pad(dem_data_final, 2, mode='edge')
smoothed_dem = np.zeros_like(dem_data_final)
for dx_offset in range(-2, 3):
    for dy_offset in range(-2, 3):
        smoothed_dem += padded_dem[2+dy_offset : 2+dy_offset+height, 2+dx_offset : 2+dx_offset+width]
smoothed_dem /= 25.0

dy_g, dx_g = np.gradient(smoothed_dem, (pixel_height * stride), (pixel_width * stride))
nz_g = np.ones_like(dx_g, dtype=np.float32)
norm_g = np.sqrt(dx_g**2 + dy_g**2 + 1.0)

vertex_normals = np.column_stack((
    -dx_g.ravel() / norm_g.ravel(),
    -dy_g.ravel() / norm_g.ravel(),
    nz_g.ravel() / norm_g.ravel()
)).astype(np.float32)

# Construct terrain Primitive and pyrender Mesh
primitive = pyrender.Primitive(
    positions=vertices,
    normals=vertex_normals,
    texcoord_0=uv,
    indices=faces,
    material=pbr_material,
    mode=pyrender.GLTF.TRIANGLES
)
pyrender_mesh = pyrender.Mesh(primitives=[primitive])

# Establish step vectors relative to peak position
peak_iy, peak_ix = np.unravel_index(np.argmax(dem_data_final), dem_data_final.shape)
step_x = 1 if peak_ix > width // 2 else -1
step_y = 1 if peak_iy > height // 2 else -1

Building terrain mesh...


In [12]:
def get_elevation_local(x, y):
    """Helper to query local terrain elevation directly from the cropped DEM."""
    idx_x = np.abs(xs_final - x).argmin()
    idx_y = np.abs(ys_final - y).argmin()
    return dem_data_final[idx_y, idx_x]

In [13]:
def create_dome_tent(radius=2.2, height=1.7, color=[0.98, 0.38, 0.05]):
    """Creates a high-resolution dome tent without bottom jaggedness."""
    sphere = trimesh.creation.icosphere(subdivisions=5, radius=1.0)
    dome = sphere.slice_plane(plane_origin=[0, 0, 0], plane_normal=[0, 0, 1])
    dome.merge_vertices()
    
    scale_mat = np.diag([radius, radius, height, 1.0])
    dome.apply_transform(scale_mat)
    
    material = pyrender.MetallicRoughnessMaterial(
        baseColorFactor=list(color) + [1.0],
        roughnessFactor=0.85,
        metallicFactor=0.0
    )
    return pyrender.Mesh.from_trimesh(dome, material=material)

In [14]:
def create_pole(height=5.0):
    """Creates a simple, robust support pole."""
    pole = trimesh.creation.cylinder(radius=0.08, height=height)
    pole.apply_translation([0, 0, height / 2.0])
    material = pyrender.MetallicRoughnessMaterial(
        baseColorFactor=[0.5, 0.45, 0.4, 1.0],
        roughnessFactor=0.9
    )
    return pyrender.Mesh.from_trimesh(pole, material=material)

In [15]:
def create_prayer_flags_string(start_pt, end_pt, num_flags=30):
    """Creates a suspended catenary line of multicolored prayer flags."""
    colors = [
        [0.1, 0.35, 0.85],   # Blue
        [0.95, 0.95, 0.95], # White
        [0.85, 0.1, 0.1],   # Red
        [0.1, 0.7, 0.25],   # Green
        [0.9, 0.8, 0.1]     # Yellow
    ]
    
    meshes = []
    t_vals = np.linspace(0.0, 1.0, num_flags)
    for i, t in enumerate(t_vals):
        pt = start_pt + t * (end_pt - start_pt)
        pt[2] -= 0.65 * np.sin(t * np.pi)
        
        flag = trimesh.creation.box(extents=[0.3, 0.01, 0.22])
        angle = np.sin(t * 12) * 0.45
        rot = trimesh.transformations.rotation_matrix(angle, [0, 0, 1])
        flag.apply_transform(rot)
        flag.apply_translation(pt)
        
        c = np.array(colors[i % len(colors)] + [1.0]) * 255
        flag.visual.vertex_colors = np.array([c] * len(flag.vertices), dtype=np.uint8)
        meshes.append(flag)
        
    combined = trimesh.util.concatenate(meshes)
    return pyrender.Mesh.from_trimesh(combined, material=None)

In [16]:
from PIL import Image

In [17]:
def generate_sample_render(sample_id, config):
    """Renders a single dataset sample (Image + Mask) from parameters."""
    # Create isolated pyrender scene
    scene = pyrender.Scene(ambient_light=config["ambient_light"])
    scene.add(pyrender_mesh)  # Reuses cached global terrain mesh
    
    # Add Directional Sun Light
    sun_light = pyrender.DirectionalLight(color=config["sun_color"], intensity=config["sun_intensity"])
    scene.add(sun_light, pose=config["sun_pose"])
    
    # Add Perspective Camera
    camera = pyrender.PerspectiveCamera(yfov=np.radians(75.0), aspectRatio=1.5, znear=1.0, zfar=150000.0)
    scene.add(camera, pose=config["cam_pose"])
    
    # Add dynamic tents and poles if requested
    for pole in config["poles"]:
        scene.add(pole["mesh"], pose=pole["pose"])
    for flag_string in config["flag_strings"]:
        scene.add(flag_string)
    for tent in config["tents"]:
        scene.add(tent["mesh"], pose=tent["pose"])
        
    # Render baseline raw geometry
    img_w, img_h = 2160, 1440
    renderer = pyrender.OffscreenRenderer(img_w, img_h)
    raw_color, depth = renderer.render(scene)
    renderer.delete()  # Crucial to prevent headless GPU memory leaks
    
    # Calculate pixel-perfect background sky mask (where depth remains unwritten)
    sky_mask = depth == 0.0
    
    # --- Generate Procedural Sky Backdrop ---
    y_idx_sky, x_idx_sky = np.indices((img_h, img_w))
    y_factor = (y_idx_sky / img_h)[:, :, np.newaxis]
    
    sky_background = (1.0 - y_factor**1.2) * config["sky_top"] + (y_factor**1.2) * config["sky_bottom"]
    
    # Apply soft sun disk glow
    dist_to_sun = np.sqrt((x_idx_sky - config["sun_img_x"])**2 + (y_idx_sky - config["sun_img_y"])**2)
    sun_glow = np.exp(-dist_to_sun / 90.0)[:, :, np.newaxis]
    sky_background += sun_glow * config["sun_glow_color"] * 0.90
    
    # Mid-day bright sun core
    if config["sun_intensity"] > 2.0:
        sun_core = (dist_to_sun < 15.0)[:, :, np.newaxis]
        sky_background = np.where(sun_core, np.array([255.0, 255.0, 255.0]), sky_background)
        
    # Blend Procedural Clouds (Multi-Octave noise)
    cloud_noise = config["cloud_l1"] * 0.55 + config["cloud_l2"] * 0.32 + config["cloud_l3"] * 0.13
    ny_sky = y_idx_sky / img_h
    cloud_height_attenuation = np.sin(ny_sky * np.pi)
    cloud_density = np.clip((cloud_noise - 0.44) * 3.2, 0.0, 1.0) * cloud_height_attenuation * config["cloud_coverage"]
    cloud_density = cloud_density[:, :, np.newaxis]
    
    sky_final = (1.0 - cloud_density) * sky_background + cloud_density * config["cloud_color"]
    sky_background_final = np.clip(sky_final, 0, 255).astype(np.uint8)
    
    # Apply atmospheric mountain haze
    extinction = np.exp(-config["fog_density"] * depth)
    extinction = np.clip(extinction, 0.0, 1.0)[:, :, np.newaxis]
    
    processed_image = (1.0 - extinction) * config["fog_color"] + extinction * raw_color.astype(np.float32)
    processed_image[sky_mask] = sky_background_final[sky_mask]
    processed_image = np.clip(processed_image, 0, 255).astype(np.uint8)
    
    # --- Simulated Lens Post-Processing ---
    img_pil = Image.fromarray(processed_image)
    
    # Subtle Chromatic Aberration
    r_chan = img_pil.getchannel('R')
    r_w, r_h = int(img_w * 1.0015), int(img_h * 1.0015)
    r_resized = r_chan.resize((r_w, r_h), Image.Resampling.LANCZOS)
    left_r, top_r = (r_w - img_w) // 2, (r_h - img_h) // 2
    r_final = r_resized.crop((left_r, top_r, left_r + img_w, top_r + img_h))
    
    g_final = img_pil.getchannel('G')
    
    b_chan = img_pil.getchannel('B')
    b_w, b_h = int(img_w * 0.9985), int(img_h * 0.9985)
    b_resized = b_chan.resize((b_w, b_h), Image.Resampling.LANCZOS)
    b_final = Image.new('L', (img_w, img_h), 0)
    left_b, top_b = (img_w - b_w) // 2, (img_h - b_h) // 2
    b_final.paste(b_resized, (left_b, top_b))
    
    camera_processed = Image.merge('RGB', (r_final, g_final, b_final))
    camera_array = np.array(camera_processed).astype(np.float32)
    
    # Wide-angle corner vignetting
    cy, cx = img_h / 2.0, img_w / 2.0
    y_grid, x_grid = np.meshgrid(np.arange(img_h), np.arange(img_w), indexing='ij')
    norm_dist_to_center = np.sqrt(((x_grid - cx) / cx)**2 + ((y_grid - cy) / cy)**2)
    vignette_mask = np.clip(1.0 - 0.35 * (norm_dist_to_center ** 2), 0.0, 1.0)[:, :, np.newaxis]
    camera_array *= vignette_mask
    
    # ISO High-Altitude Sensor Grain
    np.random.seed(sample_id)
    sensor_grain = np.random.normal(0, config["grain_intensity"], size=camera_array.shape).astype(np.float32)
    camera_array_grain = np.clip(camera_array + sensor_grain, 0, 255).astype(np.uint8)
    
    # Downsample to target training resolution (1080x720)
    final_canvas = Image.fromarray(camera_array_grain)
    final_image_resized = final_canvas.resize((1080, 720), resample=Image.Resampling.LANCZOS)
    
    # Construct binary training mask (255 = Sky, 0 = Foreground)
    binary_mask = (sky_mask * 255).astype(np.uint8)
    mask_canvas = Image.fromarray(binary_mask)
    final_mask_resized = mask_canvas.resize((1080, 720), resample=Image.Resampling.NEAREST)  # Nearest keeps mask binary
    
    return final_image_resized, final_mask_resized

In [18]:
import random
import time

In [22]:
# Generate fixed spatial structures to speed up cloud creation
img_w, img_h = 2160, 1440
np.random.seed(42)
l1_raw = np.random.uniform(0, 1, (8, 12))
l1 = np.array(Image.fromarray((l1_raw * 255).astype(np.uint8)).resize((img_w, img_h), Image.Resampling.BILINEAR)) / 255.0
l2_raw = np.random.uniform(0, 1, (24, 36))
l2 = np.array(Image.fromarray((l2_raw * 255).astype(np.uint8)).resize((img_w, img_h), Image.Resampling.BILINEAR)) / 255.0
l3_raw = np.random.uniform(0, 1, (72, 108))
l3 = np.array(Image.fromarray((l3_raw * 255).astype(np.uint8)).resize((img_w, img_h), Image.Resampling.BILINEAR)) / 255.0

grid_spacing_x = abs(xs_final[1] - xs_final[0])
grid_spacing_y = abs(ys_final[1] - ys_final[0])

# Pre-defined tent color palettes
TENT_PALETTES = [
    [0.98, 0.38, 0.05],  # Orange
    [0.98, 0.82, 0.08],  # Yellow
    [0.85, 0.1, 0.1],    # Red
    [0.1, 0.45, 0.85],   # Blue
    [0.15, 0.65, 0.25]   # Green
]

# CRITICAL FIX: Self-correcting fallback check
if 'dem_width' not in globals() or 'dem_height' not in globals():
    dem_height, dem_width = dem_data_final.shape

# Calculate dynamic margins based on robust grid dimensions
margin_x = max(1, min(15, dem_width // 4))
margin_y = max(1, min(15, dem_height // 4))

print(f"Beginning plausible regional rendering loop. Targeted: {TOTAL_SAMPLES} samples.")

for sample_id in range(TOTAL_SAMPLES):
    if sample_id in completed_ids:
        continue  # Skip already generated samples on resume
        
    print(f"Generating sample {sample_id:03d} / {TOTAL_SAMPLES}...")
    
    # Initialize deterministic local generator for this sample ID
    rng = np.random.default_rng(sample_id)
    
    # 1. Determine local Base Camp flat position using safe global grid variables
    best_ix, best_iy = peak_ix - 45 * step_x, peak_iy - 45 * step_y
    min_slope = 999.0
    r_step = rng.choice(list(range(55, 90, 5)))
    for angle in np.linspace(0, 2 * np.pi, 16):
        ix = int(peak_ix + r_step * np.cos(angle))
        iy = int(peak_iy + r_step * np.sin(angle))
        if margin_x <= ix < dem_width - margin_x and margin_y <= iy < dem_height - margin_y:
            next_ix = ix + 1
            next_iy = iy + 1
            dz_dx = (dem_data_final[iy, next_ix] - dem_data_final[iy, ix]) / grid_spacing_x
            dz_dy = (dem_data_final[next_iy, ix] - dem_data_final[iy, ix]) / grid_spacing_y
            slope = np.sqrt(dz_dx**2 + dz_dy**2)
            if slope < min_slope:
                min_slope = slope
                best_ix, best_iy = ix, iy
                
    cam_ground_z = dem_data_final[best_iy, best_ix]
    eye_x = xs_final[best_ix]
    eye_y = ys_final[best_iy]
    
    # Randomize Camera Height (4.5m to 9.0m above ground)
    eye_z = cam_ground_z + rng.uniform(4.5, 9.0)
    eye = np.array([eye_x, eye_y, eye_z], dtype=np.float32)
    
    # Map the peak coordinates to the physical space
    peak_x = xs_final[peak_ix]
    peak_y = ys_final[peak_iy]
    peak_z = dem_data_final[peak_iy, peak_ix]
    
    dir_to_peak = np.array([peak_x - eye[0], peak_y - eye[1], 0.0])
    dir_to_peak /= np.linalg.norm(dir_to_peak)
    
    # Random camera direction pointing around the peaks
    target = eye + dir_to_peak * 200.0 + np.array([
        rng.uniform(-25.0, 25.0),
        rng.uniform(-25.0, 25.0),
        rng.uniform(15.0, 35.0)
    ], dtype=np.float32)
    
    # Compute pose
    forward = target - eye
    forward_norm = forward / np.linalg.norm(forward)
    forward_horiz = np.array([forward_norm[0], forward_norm[1], 0.0])
    forward_horiz /= np.linalg.norm(forward_horiz)
    up_global = np.array([0.0, 0.0, 1.0], dtype=np.float32)
    right = np.cross(forward_horiz, up_global)
    right /= np.linalg.norm(right)
    up_camera = np.cross(right, forward_norm)
    up_camera /= np.linalg.norm(up_camera)
    
    cam_pose = np.eye(4, dtype=np.float32)
    cam_pose[:3, 0] = right
    cam_pose[:3, 1] = up_camera
    cam_pose[:3, 2] = -forward_norm
    cam_pose[:3, 3] = eye
    
    # 2. Randomize Environmental weather types
    weather = rng.choice(["sunny", "overcast", "sunset", "stormy_haze"])
    
    # Procedurally construct randomized sun orientations
    azimuth = rng.uniform(-np.pi, np.pi)
    elevation = rng.uniform(np.radians(20.0), np.radians(70.0))
    
    z_axis = np.array([
        np.cos(elevation) * np.sin(azimuth),
        np.cos(elevation) * np.cos(azimuth),
        np.sin(elevation)
    ], dtype=np.float32)
    z_axis /= np.linalg.norm(z_axis)
    
    x_axis = np.cross([0, 0, 1], z_axis)
    if np.linalg.norm(x_axis) < 1e-5:
        x_axis = np.array([1.0, 0.0, 0.0], dtype=np.float32)
    else:
        x_axis /= np.linalg.norm(x_axis)
    y_axis = np.cross(z_axis, x_axis)
    y_axis /= np.linalg.norm(y_axis)
    
    sun_pose = np.eye(4, dtype=np.float32)
    sun_pose[:3, 0] = x_axis
    sun_pose[:3, 1] = y_axis
    sun_pose[:3, 2] = z_axis
    
    # Default configs
    config = {
        "cam_pose": cam_pose,
        "sun_pose": sun_pose,
        "cloud_l1": l1, "cloud_l2": l2, "cloud_l3": l3,
        "grain_intensity": rng.uniform(1.5, 4.0),
        "poles": [], "flag_strings": [], "tents": []
    }
    
    if weather == "sunny":
        config["ambient_light"] = [0.18, 0.20, 0.28]
        config["sun_color"] = [1.0, 1.0, 0.95]
        config["sun_intensity"] = rng.uniform(2.8, 3.8)
        config["sky_top"] = np.array([10.0, 45.0, 130.0])
        config["sky_bottom"] = np.array([170.0, 210.0, 245.0])
        config["sun_glow_color"] = np.array([255.0, 245.0, 215.0])
        config["cloud_coverage"] = rng.uniform(0.0, 0.40)
        config["cloud_color"] = np.array([245.0, 248.0, 252.0])
        config["fog_density"] = rng.uniform(0.000005, 0.000018)
        config["fog_color"] = np.array([165, 185, 210], dtype=np.float32)
    elif weather == "overcast":
        config["ambient_light"] = [0.22, 0.22, 0.22]
        config["sun_color"] = [0.90, 0.90, 0.90]
        config["sun_intensity"] = rng.uniform(0.8, 1.4)
        config["sky_top"] = np.array([80.0, 85.0, 95.0])
        config["sky_bottom"] = np.array([160.0, 165.0, 175.0])
        config["sun_glow_color"] = np.array([180.0, 180.0, 180.0])
        config["cloud_coverage"] = rng.uniform(0.65, 0.95)
        config["cloud_color"] = np.array([210.0, 212.0, 215.0])
        config["fog_density"] = rng.uniform(0.000030, 0.000055)
        config["fog_color"] = np.array([180, 185, 195], dtype=np.float32)
    elif weather == "sunset":
        config["ambient_light"] = [0.12, 0.08, 0.18]
        config["sun_color"] = [1.0, 0.45, 0.15]
        config["sun_intensity"] = rng.uniform(1.8, 2.6)
        config["sky_top"] = np.array([25.0, 15.0, 60.0])
        config["sky_bottom"] = np.array([255.0, 110.0, 50.0])
        config["sun_glow_color"] = np.array([255.0, 160.0, 70.0])
        config["cloud_coverage"] = rng.uniform(0.15, 0.70)
        config["cloud_color"] = np.array([230.0, 110.0, 90.0])
        config["fog_density"] = rng.uniform(0.000010, 0.000030)
        config["fog_color"] = np.array([210, 140, 120], dtype=np.float32)
    elif weather == "stormy_haze":
        config["ambient_light"] = [0.15, 0.16, 0.18]
        config["sun_color"] = [0.95, 0.90, 0.85]
        config["sun_intensity"] = rng.uniform(1.2, 2.0)
        config["sky_top"] = np.array([45.0, 45.0, 50.0])
        config["sky_bottom"] = np.array([120.0, 120.0, 125.0])
        config["sun_glow_color"] = np.array([140.0, 135.0, 125.0])
        config["cloud_coverage"] = rng.uniform(0.40, 0.85)
        config["cloud_color"] = np.array([130.0, 130.0, 135.0])
        config["fog_density"] = rng.uniform(0.000040, 0.000075)
        config["bottom_fog_inc"] = 0.00002
        config["fog_color"] = np.array([140, 145, 150], dtype=np.float32)
        
    config["sun_img_x"] = int(img_w * rng.uniform(0.20, 0.80))
    config["sun_img_y"] = int(img_h * rng.uniform(0.10, 0.35))
    
    # 3. Randomize prayer flag presence
    if rng.random() < 0.65:
        p1_x = eye[0] + rng.uniform(3.5, 5.5) * forward_horiz[0] - rng.uniform(1.5, 3.5) * right[0]
        p1_y = eye[1] + rng.uniform(3.5, 5.5) * forward_horiz[1] - rng.uniform(1.5, 3.5) * right[1]
        p1_z = get_elevation_local(p1_x, p1_y)
        
        p2_x = eye[0] + rng.uniform(14.0, 18.0) * forward_horiz[0] + rng.uniform(2.5, 5.5) * right[0]
        p2_y = eye[1] + rng.uniform(14.0, 18.0) * forward_horiz[1] + rng.uniform(2.5, 5.5) * right[1]
        p2_z = get_elevation_local(p2_x, p2_y)
        
        pole_h = rng.uniform(4.5, 5.5)
        config["poles"].append({"mesh": create_pole(height=pole_h), "pose": np.array([
            [1, 0, 0, p1_x], [0, 1, 0, p1_y], [0, 0, 1, p1_z], [0, 0, 0, 1]
        ])})
        config["poles"].append({"mesh": create_pole(height=pole_h), "pose": np.array([
            [1, 0, 0, p2_x], [0, 1, 0, p2_y], [0, 0, 1, p2_z], [0, 0, 0, 1]
        ])})
        
        config["flag_strings"].append(create_prayer_flags_string(
            np.array([p1_x, p1_y, p1_z + pole_h - 0.2]),
            np.array([p2_x, p2_y, p2_z + pole_h - 0.2]),
            num_flags=rng.integers(28, 36)
        ))
        
    # 4. Generate randomized, non-overlapping coordinates for tents
    num_tents = rng.integers(2, 7)
    tent_offsets = []
    attempts = 0
    while len(tent_offsets) < num_tents and attempts < 150:
        dist = rng.uniform(9.0, 24.0)
        lat = rng.uniform(-6.0, 6.0)
        
        intersect = False
        for p_dist, p_lat in tent_offsets:
            if np.sqrt((dist - p_dist)**2 + (lat - p_lat)**2) < 4.8:
                intersect = True
                break
        if not intersect:
            tent_offsets.append((dist, lat))
        attempts += 1
        
    for dist, lat in tent_offsets:
        tx = eye[0] + dist * forward_horiz[0] + lat * right[0]
        ty = eye[1] + dist * forward_horiz[1] + lat * right[1]
        
        t_ix = np.abs(xs_final - tx).argmin()
        t_iy = np.abs(ys_final - ty).argmin()
        
        # Calculate step vector using safe variables
        step_dir_x = 1 if np.sin(yaw) >= 0 else -1
        step_dir_y = 1 if np.cos(yaw) >= 0 else -1
        
        next_ix = t_ix + step_dir_x if (0 <= t_ix + step_dir_x < dem_width) else t_ix - step_dir_x
        next_iy = t_iy + step_dir_y if (0 <= t_iy + step_dir_y < dem_height) else t_iy - step_dir_y
        
        dz_dx = (dem_data_final[t_iy, next_ix] - dem_data_final[t_iy, t_ix]) / ((next_ix - t_ix) * grid_spacing_x)
        dz_dy = (dem_data_final[next_iy, t_ix] - dem_data_final[t_iy, t_ix]) / ((next_iy - t_iy) * grid_spacing_y)
        
        slope_mag = np.sqrt(dz_dx**2 + dz_dy**2)
        slope_mag = np.clip(slope_mag, 0.0, 0.45) # Clamp to prevent extreme sinks on raw cliffs
        
        tz = dem_data_final[t_iy, t_ix] - (slope_mag * 2.2) - 0.10
        
        color = rng.choice(TENT_PALETTES)
        radius = rng.uniform(2.0, 2.3)
        tent_height = rng.uniform(1.5, 1.8)  # Safe local name
        tent_pose = np.eye(4)
        tent_pose[:3, 3] = [tx, ty, tz]
        
        config["tents"].append({
            "mesh": create_dome_tent(radius=radius, height=tent_height, color=color),
            "pose": tent_pose
        })
        
    # 5. Run render pass and record output
    try:
        final_img, final_mask = generate_sample_render(sample_id, config)
        
        # Save results to disk
        img_filename = f"sample_{sample_id:04d}.png"
        final_img.save(os.path.join(IMAGES_DIR, img_filename))
        final_mask.save(os.path.join(MASKS_DIR, img_filename))
        
        # Update and save progress
        completed_ids.append(sample_id)
        save_progress(completed_ids)
        
    except Exception as e:
        print(f"Skipping failed render pass on sample {sample_id} to avoid crash: {e}")
        time.sleep(0.5)

print(f"Generation pass complete. Total images completed: {len(completed_ids)} / {TOTAL_SAMPLES}")

Beginning plausible regional rendering loop. Targeted: 500 samples.
Generating sample 001 / 500...
Generating sample 002 / 500...
Generating sample 003 / 500...
Generating sample 004 / 500...
Generating sample 005 / 500...
Generating sample 006 / 500...
Generating sample 007 / 500...
Generating sample 008 / 500...
Generating sample 009 / 500...
Generating sample 010 / 500...
Generating sample 011 / 500...
Generating sample 012 / 500...
Generating sample 013 / 500...
Generating sample 014 / 500...
Generating sample 015 / 500...
Generating sample 016 / 500...
Generating sample 017 / 500...
Generating sample 018 / 500...
Generating sample 019 / 500...
Generating sample 020 / 500...
Generating sample 021 / 500...
Generating sample 022 / 500...
Generating sample 023 / 500...
Generating sample 024 / 500...
Generating sample 025 / 500...
Generating sample 026 / 500...
Generating sample 027 / 500...
Generating sample 028 / 500...
Generating sample 029 / 500...
Generating sample 030 / 500...
Ge